# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Adapted from your existing exercise. Previous local outputs were cleared so new Colab runs provide their own evidence. 

Source: Original local package: 03_fastai/00_image_classifier.ipynb

In [1]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '03_fastai', 'fastai', '00_image_classifier.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

Exercise folder: /content/HaiLong_Colab_Exercises/03_fastai
Run mode: FULL EXERCISE
Results: /content/drive/MyDrive/HaiLong_Colab_Results/03_fastai/results


## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.
- Use the real photographic branch and retain its split, class counts, held-out results and model artefacts.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.


# Lesson 1: build an image classifier
Source task: https://github.com/fastai/course22/blob/master/00-is-it-a-bird-creating-a-model-from-your-own-data.ipynb
This is an independently written solution companion, not a copy of the instructor notebook or a course completion record. The recorded CPU smoke run uses clearly named synthetic pattern images to verify training and export. The full branch implements the photographic bird/forest exercise and needs internet access or your local image folders.


In [2]:
import os,json
from pathlib import Path
import numpy as np,pandas as pd,torch
import matplotlib.pyplot as plt
torch.manual_seed(42);torch.set_num_threads(1)
np.random.seed(42)
SMOKE=os.environ.get('COURSE_SMOKE','0')=='1'
results=Path('results');results.mkdir(exist_ok=True)

from fastai.vision.all import *
from PIL import Image
if SMOKE:
    path=Path('data/synthetic_image_smoke')
    rng=np.random.default_rng(42)
    for label in ['horizontal_pattern','vertical_pattern']:
        folder=path/label;folder.mkdir(parents=True,exist_ok=True)
        for i in range(24):
            yy,xx=np.mgrid[:64,:64];phase=rng.uniform(0,6)
            pattern=np.sin((yy if label=='horizontal_pattern' else xx)/5+phase)
            pixels=np.clip(120+80*pattern[:,:,None]+rng.normal(0,8,(64,64,3)),0,255).astype('uint8')
            Image.fromarray(pixels).save(folder/f'{i:03d}.png')
    print('SYNTHETIC IMAGE SMOKE TEST; these are not bird or forest photographs.')
else:
    path=Path('data/bird_forest')
    from ddgs import DDGS
    import hashlib,tempfile,shutil
    # An earlier failed download can leave the root and only one class folder.
    # Attempt the original 60-result search for each class on every full retry.
    # Merge downloads by content so retries keep photographs without duplicates.
    for label,query in [('bird','bird photo'),('forest','forest photo')]:
        folder=path/label;folder.mkdir(parents=True,exist_ok=True)
        matches=DDGS().images(query,max_results=60)
        urls=[item['image'] for item in matches]
        existing_hashes={hashlib.sha256(f.read_bytes()).hexdigest() for f in get_image_files(folder)}
        with tempfile.TemporaryDirectory(prefix='course_image_download_') as stage:
            download_images(Path(stage),urls=urls,n_workers=4)
            for downloaded in get_image_files(stage):
                digest=hashlib.sha256(downloaded.read_bytes()).hexdigest()
                if digest not in existing_hashes:
                    shutil.copy2(downloaded,folder/f'{digest}{downloaded.suffix.lower()}')
                    existing_hashes.add(digest)
    print('Photographic course task:',path)
files=get_image_files(path)
# Retain only readable images; leave source files untouched.
failed=set(verify_images(files));files=L([f for f in files if f not in failed])
assert len(files)>10
assert set(parent_label(f) for f in files) == {'horizontal_pattern','vertical_pattern'} if SMOKE else set(parent_label(f) for f in files) == {'bird','forest'}
from collections import Counter
class_counts=Counter(parent_label(f) for f in files)
assert min(class_counts.values()) >= 5, f'Insufficient readable images per class: {class_counts}'
# Persist the actual inputs so export/reload and comparison survive a fresh runtime.
import shutil
persistent_images=results/('image_dataset_smoke' if SMOKE else 'image_dataset_full')
for source_file in files:
    target=persistent_images/parent_label(source_file)/source_file.name
    target.parent.mkdir(parents=True,exist_ok=True)
    if not target.exists(): shutil.copy2(source_file,target)
pd.DataFrame({'image':[str(f) for f in files],'label':[parent_label(f) for f in files]}).to_csv(results/'image_manifest.csv',index=False)
block=DataBlock(blocks=(ImageBlock,CategoryBlock),get_items=lambda _:files,
    splitter=RandomSplitter(valid_pct=.2,seed=42),get_y=parent_label,
    item_tfms=Resize(64 if SMOKE else 192))
dls=block.dataloaders(path,bs=8,num_workers=0)
print('Classes:',dls.vocab,'; train:',len(dls.train_ds),'; validation:',len(dls.valid_ds))

Photographic course task: data/bird_forest


Classes: ['bird', 'forest'] ; train: 52 ; validation: 13


In [3]:
learn=vision_learner(dls,resnet18,pretrained=not SMOKE,metrics=error_rate)
with learn.no_bar():learn.fine_tune(1 if SMOKE else 3)
probs,targets=learn.get_preds(dl=dls.valid)
targets=targets.reshape(-1)
validation_accuracy=float((probs.argmax(1)==targets).float().mean())
learn.export((results/'image_classifier.pkl').resolve())
label,index,probabilities=learn.predict(files[0])
print({'predicted_label':str(label),'validation_accuracy':validation_accuracy,'synthetic_smoke':SMOKE})
(results/'image_config.json').write_text(json.dumps({'seed':42,'synthetic_smoke':SMOKE,
    'labels':list(map(str,dls.vocab)),'validation_accuracy':validation_accuracy,
    'class_counts':dict(class_counts),'readable_images':len(files),'fine_tune_epochs':1 if SMOKE else 3},indent=2))
assert torch.isfinite(probs).all()
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(targets.numpy(),probs.argmax(1).numpy())
pd.DataFrame(cm,index=list(dls.vocab),columns=list(dls.vocab)).to_csv(results/'image_confusion_matrix.csv')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


  0%|          | 0.00/44.7M [00:00<?, ?B/s]

 24%|██▎       | 10.5M/44.7M [00:00<00:00, 110MB/s]

 62%|██████▏   | 27.5M/44.7M [00:00<00:00, 150MB/s]

100%|██████████| 44.7M/44.7M [00:00<00:00, 158MB/s]

...

Storage check (checkpoint_write): artifacts 2.374 GB; configured budget 9.000 GB; additional estimate 0.196 GB; runtime free 75.664 GB. Budget is not Drive quota.


[2, 0.12627831101417542, 0.0002587837225291878, 0.0, '00:12']


Storage check (checkpoint_write): artifacts 2.374 GB; configured budget 9.000 GB; additional estimate 0.196 GB; runtime free 75.664 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 2.374 GB; configured budget 9.000 GB; additional estimate 0.055 GB; runtime free 75.476 GB. Budget is not Drive quota.


{'predicted_label': 'forest', 'validation_accuracy': 1.0, 'synthetic_smoke': False}


Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [4]:
finish_run(COLAB_CONTEXT)

Saved run settings. Save the notebook in Colab to keep its cell outputs.


In [5]:
# Added by the course runner: inspect the executed runtime, not coursework.
import os as _hailong_os, json as _hailong_json, platform as _hailong_platform
_hailong_evidence = {
    'python': _hailong_platform.python_version(),
    'quick_run': bool(globals().get('QUICK_RUN', False)),
    'course_smoke': _hailong_os.environ.get('COURSE_SMOKE', '0'),
    'keras_backend': _hailong_os.environ.get('KERAS_BACKEND'),
    'legacy_keras': _hailong_os.environ.get('TF_USE_LEGACY_KERAS'),
    'base_model_commit': getattr(getattr(globals().get('model'), 'config', None), '_commit_hash', None),
}
_hailong_context = globals().get('COLAB_CONTEXT')
if _hailong_context and _hailong_context['folder'].relative_to(_hailong_context['root']).as_posix().startswith(('03_fastai', '04_fastai_part2')):
    from course_persistence import collect_reuse_evidence as _hailong_collect
    _hailong_evidence['fastai_reuse_evidence'] = _hailong_context['run'].get('reuse_evidence') or _hailong_collect(_hailong_context)
print('__HAILONG_RUNTIME_EVIDENCE__' + _hailong_json.dumps(_hailong_evidence))


__HAILONG_RUNTIME_EVIDENCE__{"python": "3.12.13", "quick_run": false, "course_smoke": "0", "keras_backend": null, "legacy_keras": null, "base_model_commit": null, "fastai_reuse_evidence": {"contract_version": 1, "notebook": "03_fastai/00_image_classifier.ipynb", "source_sha256": "99ad6768629fe5595746179c2be4d60a2c23402e705fb5b740128a26e9437041", "helper_sha256": {"03_fastai/convolution_workbook_exercise.py": "ecacb93aef6846c82d6a9eebc4809e1a6766410f04783ac132f247935af3a8fc", "03_fastai/fastai_utils.py": "a03e913e1c07f55aedd56ee6128096f3f952f4f8be22f83c4a22002589c4bd18", "03_fastai/wearable_objective_extension.py": "f9b0a8bf81a5bab8c10aee5e8795e9cef3df58e40925f7705e58301b18b3fbe9", "03_fastai/wearable_utils.py": "6de4ee45118d96e024bd4e0f3fd3552905b38a1ff01c0a5146578bb0d8263db9", "colab_runtime.py": "e11bedb13e1bd71a71c01cc49e558d05c89b057259fe52e64301d0c4d626a140", "course_persistence.py": "f14f7282d4b1e718b01858a61a2dca8b60f5135ba9971fc7923c50bf12cca3d2"}, "persistent_course_root": "/c